# 05b - Configuration-Driven Pipelines: Advanced

This notebook covers advanced configuration features:

| Part | Topic |
|------|-------|
| **3** | Custom Transformers & Functions |
| **4** | Lazy Parameters (`__ns__`) |
| **5** | Loops (Dynamic Expansion) |
| **6** | Full Examples |

**Prerequisites:** Notebook 05a (basic configuration format)

In [1]:
import polars as pl

from nebula.base import Transformer
from nebula.pipelines.pipeline_loader import load_pipeline
from nebula.storage import nebula_storage as ns

In [2]:
orders = pl.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer": ["alice", "bob", "alice", "carol", "bob"],
    "amount": [150.0, 75.0, 200.0, 50.0, 300.0],
    "region": ["US", "EU", "US", "APAC", "EU"],
})
orders

order_id,customer,amount,region
i64,str,f64,str
1,"""alice""",150.0,"""US"""
2,"""bob""",75.0,"""EU"""
3,"""alice""",200.0,"""US"""
4,"""carol""",50.0,"""APAC"""
5,"""bob""",300.0,"""EU"""


---
## Part 3: Custom Transformers & Functions

To use transformers not in Nebula's core, pass them via `extra_transformers`. For split/branch functions, use `extra_functions`.

**Best practice:** Use `dict[str, T]` where the **key is the contract** with the config. This is refactor-safe - you can rename Python classes/functions without breaking configs.

### 3.1 Extra Transformers

In [3]:
# Define custom transformers
class AddProcessedFlag(Transformer):
    def _transform_nw(self, df):
        import narwhals as nw
        return df.with_columns(nw.lit(True).alias("processed"))


class DoubleAmount(Transformer):
    def _transform_nw(self, df):
        import narwhals as nw
        return df.with_columns((nw.col("amount") * 2).alias("amount"))


# Config references transformers by name (the dict key)
config = {
    "pipeline": [
        {"transformer": "AddProcessedFlag"},
        {"transformer": "DoubleAmount"},
    ]
}

# Pass custom transformers as dict[str, class]
# Keys are the names used in config - decoupled from Python class names
pipe = load_pipeline(
    config,
    extra_transformers={
        "AddProcessedFlag": AddProcessedFlag,
        "DoubleAmount": DoubleAmount,
    }
)

result = pipe.run(orders.head(2))
print(result)

2026-03-09 12:46:43,715 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,717 | [INFO]: Running 'AddProcessedFlag' ... 
2026-03-09 12:46:43,726 | [INFO]: Completed 'AddProcessedFlag' in 0.0s 
2026-03-09 12:46:43,726 | [INFO]: Running 'DoubleAmount' ... 
2026-03-09 12:46:43,726 | [INFO]: Completed 'DoubleAmount' in 0.0s 
2026-03-09 12:46:43,731 | [INFO]: Pipeline completed in 0.0s 


shape: (2, 5)
┌──────────┬──────────┬────────┬────────┬───────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ processed │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---       │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ bool      │
╞══════════╪══════════╪════════╪════════╪═══════════╡
│ 1        ┆ alice    ┆ 300.0  ┆ US     ┆ true      │
│ 2        ┆ bob      ┆ 150.0  ┆ EU     ┆ true      │
└──────────┴──────────┴────────┴────────┴───────────┘


**Why dict keys matter for refactoring:**

```python
# If you rename DoubleAmount → MultiplyAmount in Python:
extra_transformers={
    "DoubleAmount": MultiplyAmount,  # Key stays same, config doesn't break
}
```

### 3.2 Using Modules

For production, organize transformers in modules:

```python
# my_transformers.py
__all__ = ["AddProcessedFlag", "DoubleAmount"]

class AddProcessedFlag(Transformer): ...
class DoubleAmount(Transformer): ...
```

```python
# Load from module
import my_transformers

pipe = load_pipeline(config, extra_transformers=[my_transformers])
```

### 3.3 Extra Functions (for Split/Branch)

In [4]:
def split_by_region(df):
    """Split DataFrame by region."""
    return {
        "us": df.filter(pl.col("region") == "US"),
        "international": df.filter(pl.col("region") != "US"),
    }


config = {
    "split_function": "split_by_region",  # References dict key
    "pipeline": {
        "us": [
            {"transformer": "AddLiterals", "params": {
                "data": [{"alias": "tax_rate", "value": 0.08}]
            }}
        ],
        "international": [
            {"transformer": "AddLiterals", "params": {
                "data": [{"alias": "tax_rate", "value": 0.20}]
            }}
        ],
    }
}

pipe = load_pipeline(
    config,
    extra_functions={
        "split_by_region": split_by_region,  # Key matches config reference
    }
)

pipe.show()
result = pipe.run(orders)
print(result)

2026-03-09 12:46:43,747 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,752 | [INFO]: Entering split 
2026-03-09 12:46:43,756 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:43,758 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:43,759 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:43,760 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:43,762 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
------ SPLIT ------ (function: split_by_region)
**SPLIT <<< international >>> (1 transformation):
     - AddLiterals
**SPLIT <<< us >>> (1 transformation):
     - AddLiterals
<<< Append DFs >>>
shape: (5, 5)
┌──────────┬──────────┬────────┬────────┬──────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ tax_rate │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---      │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ f64      │
╞══════════╪══════════╪════════╪════════╪══════════╡
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ 0.2      │
│ 4        ┆ carol    ┆ 50.0   ┆ APAC   ┆ 0.2      │
│ 5        ┆ bob      ┆ 300.0  ┆ EU     ┆ 0.2      │
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ 0.08     │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ 0.08     │
└──────────┴──────────┴────────┴────────┴──────────┘


### 3.4 Function Dicts (Inline Callables)

Alongside `transformer` entries, you can embed **arbitrary Python functions** directly in a config pipeline using the `function` key:

| Key | Required | Description |
|-----|----------|-------------|
| `function` | Yes | A callable **or** a string name registered in `extra_functions` |
| `args` | No | Positional arguments passed to the function after `df` |
| `kwargs` | No | Keyword arguments passed to the function |
| `description` | No | Human-readable label shown by `pipeline.show()` |
| `skip` | No | If `true`, skip this step |
| `perform` | No | If `false`, skip this step |

This bridges **declarative config** with **imperative logic** — great for one-off transformations that don't warrant a full `Transformer` class.

#### 3.4.1 Direct Callable (Python Dict API)

In [5]:
import polars as pl

# Any regular function that accepts a DataFrame and returns a DataFrame
def tag_source(df):
    """Add a static source tag."""
    return df.with_columns(pl.lit("orders_v2").alias("source"))


config = [
    {"transformer": "SelectColumns", "params": {"columns": ["order_id", "amount"]}},
    # Pass the callable directly – no string lookup needed
    {"function": tag_source, "description": "Tag the data source"},
]

pipe = load_pipeline(config)
pipe.show(add_params=True)

result = pipe.run(orders)
print(result)

2026-03-09 12:46:43,776 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,776 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:43,780 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:43,780 | [INFO]: Running 'tag_source' ... 
2026-03-09 12:46:43,780 | [INFO]: Completed 'tag_source' in 0.0s 
2026-03-09 12:46:43,780 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
 - SelectColumns -> PARAMS: columns=['order_id', 'amount']
 - tag_source
     Description: Tag the data source
shape: (5, 3)
┌──────────┬────────┬───────────┐
│ order_id ┆ amount ┆ source    │
│ ---      ┆ ---    ┆ ---       │
│ i64      ┆ f64    ┆ str       │
╞══════════╪════════╪═══════════╡
│ 1        ┆ 150.0  ┆ orders_v2 │
│ 2        ┆ 75.0   ┆ orders_v2 │
│ 3        ┆ 200.0  ┆ orders_v2 │
│ 4        ┆ 50.0   ┆ orders_v2 │
│ 5        ┆ 300.0  ┆ orders_v2 │
└──────────┴────────┴───────────┘


#### 3.4.2 String Name Lookup

In YAML/JSON configs you can't embed callables, so you use a **string name** that is resolved against `extra_functions` at load time — the same registry used for split functions.

```yaml
# pipeline.yaml
pipeline:
  - transformer: SelectColumns
    params: {columns: [order_id, amount]}
  - function: tag_source          # ← resolved from extra_functions
    description: Tag the data source
```

In [6]:
config = [
    {"transformer": "SelectColumns", "params": {"columns": ["order_id", "amount"]}},
    {"function": "tag_source", "description": "Tag the data source"},
]

pipe = load_pipeline(
    config,
    extra_functions={"tag_source": tag_source},  # key = name used in config
)
pipe.show(add_params=True)

result = pipe.run(orders)
print(result)

2026-03-09 12:46:43,802 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,802 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:43,805 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:43,805 | [INFO]: Running 'tag_source' ... 
2026-03-09 12:46:43,807 | [INFO]: Completed 'tag_source' in 0.0s 
2026-03-09 12:46:43,808 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
 - SelectColumns -> PARAMS: columns=['order_id', 'amount']
 - tag_source
     Description: Tag the data source
shape: (5, 3)
┌──────────┬────────┬───────────┐
│ order_id ┆ amount ┆ source    │
│ ---      ┆ ---    ┆ ---       │
│ i64      ┆ f64    ┆ str       │
╞══════════╪════════╪═══════════╡
│ 1        ┆ 150.0  ┆ orders_v2 │
│ 2        ┆ 75.0   ┆ orders_v2 │
│ 3        ┆ 200.0  ┆ orders_v2 │
│ 4        ┆ 50.0   ┆ orders_v2 │
│ 5        ┆ 300.0  ┆ orders_v2 │
└──────────┴────────┴───────────┘


#### 3.4.3 Passing Args and Kwargs

In [7]:
def apply_discount(df, col, rate=0.1):
    """Multiply a column by (1 - rate)."""
    return df.with_columns(
        (pl.col(col) * (1 - rate)).alias(col)
    )


def add_batch_tag(df, batch_id, env="prod"):
    """Stamp every row with batch metadata."""
    return df.with_columns(
        pl.lit(batch_id).alias("batch_id"),
        pl.lit(env).alias("env"),
    )


config = [
    # args: positional arguments after df
    {
        "function": "apply_discount",
        "args": ["amount"],          # col = 'amount'
        "kwargs": {"rate": 0.15},     # rate = 0.15
        "description": "Apply 15% discount to amount",
    },
    # kwargs only
    {
        "function": "add_batch_tag",
        "kwargs": {"batch_id": "batch_20260227", "env": "staging"},
        "description": "Stamp batch metadata",
    },
]

pipe = load_pipeline(
    config,
    extra_functions={"apply_discount": apply_discount, "add_batch_tag": add_batch_tag},
)
pipe.show(add_params=True)

result = pipe.run(orders)
print(result)

2026-03-09 12:46:43,830 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,830 | [INFO]: Running 'apply_discount' ... 
2026-03-09 12:46:43,830 | [INFO]: Completed 'apply_discount' in 0.0s 
2026-03-09 12:46:43,834 | [INFO]: Running 'add_batch_tag' ... 
2026-03-09 12:46:43,834 | [INFO]: Completed 'add_batch_tag' in 0.0s 
2026-03-09 12:46:43,835 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
 - apply_discount -> ARGS=['amount'], KWARGS={'rate': 0.15}
     Description: Apply 15% discount to amount
 - add_batch_tag -> KWARGS={'batch_id': 'batch_20260227', 'env': 'staging'}
     Description: Stamp batch metadata
shape: (5, 6)
┌──────────┬──────────┬────────┬────────┬────────────────┬─────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ batch_id       ┆ env     │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---            ┆ ---     │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str            ┆ str     │
╞══════════╪══════════╪════════╪════════╪════════════════╪═════════╡
│ 1        ┆ alice    ┆ 127.5  ┆ US     ┆ batch_20260227 ┆ staging │
│ 2        ┆ bob      ┆ 63.75  ┆ EU     ┆ batch_20260227 ┆ staging │
│ 3        ┆ alice    ┆ 170.0  ┆ US     ┆ batch_20260227 ┆ staging │
│ 4        ┆ carol    ┆ 42.5   ┆ APAC   ┆ batch_20260227 ┆ staging │
│ 5        ┆ bob      ┆ 255.0  ┆ EU     ┆ batch_20260227 ┆ staging │
└──────────┴──────────┴────────┴─────

#### 3.4.4 Skip / Perform Flags

Function steps support the same `skip` / `perform` flags as transformers — ideal for feature flags and debug steps.

In [8]:
DEBUG_MODE = False  # flip to True to enable the debug step


def log_shape(df):
    """Debug helper: print shape and return df unchanged."""
    print(f"[DEBUG] shape={df.shape}, columns={df.columns}")
    return df


config = [
    {"transformer": "Filter", "params": {
        "input_col": "amount", "perform": "keep", "operator": "gt", "value": 100
    }},
    # This step only runs when DEBUG_MODE is True
    {
        "function": "log_shape",
        "perform": DEBUG_MODE,
        "description": "Debug: log shape after filter",
    },
    {"transformer": "SelectColumns", "params": {"columns": ["order_id", "customer", "amount"]}},
]

pipe = load_pipeline(config, extra_functions={"log_shape": log_shape})
pipe.show()  # log_shape is absent from the shown pipeline

result = pipe.run(orders)
print(result)

2026-03-09 12:46:43,858 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,858 | [INFO]: Running 'Filter' ... 
2026-03-09 12:46:43,863 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:46:43,863 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:43,865 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:43,866 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
 - Filter
 - SelectColumns
shape: (3, 3)
┌──────────┬──────────┬────────┐
│ order_id ┆ customer ┆ amount │
│ ---      ┆ ---      ┆ ---    │
│ i64      ┆ str      ┆ f64    │
╞══════════╪══════════╪════════╡
│ 1        ┆ alice    ┆ 150.0  │
│ 3        ┆ alice    ┆ 200.0  │
│ 5        ┆ bob      ┆ 300.0  │
└──────────┴──────────┴────────┘


#### 3.4.5 Mixed Pipeline — Transformers + Functions

Functions and transformers compose freely. This is a realistic ETL step where a custom normalisation function lives alongside standard Nebula transformers:

In [9]:
def normalize_amounts(df, decimals=2):
    """Round amounts and cap at a maximum value."""
    return df.with_columns(
        pl.col("amount").round(decimals).clip(upper_bound=250.0).alias("amount")
    )


def add_audit_cols(df, run_id):
    """Append audit columns required by the data warehouse."""
    import datetime
    return df.with_columns(
        pl.lit(run_id).alias("run_id"),
        pl.lit(str(datetime.date.today())).alias("load_date"),
    )


config = {
    "name": "Order ETL",
    "pipeline": [
        # 1. Validate input
        {"transformer": "AssertNotEmpty"},
        # 2. Keep relevant columns
        {"transformer": "SelectColumns",
         "params": {"columns": ["order_id", "customer", "amount", "region"]}},
        # 3. Custom normalisation logic (function, no full Transformer needed)
        {"function": "normalize_amounts",
         "kwargs": {"decimals": 2},
         "description": "Round & cap amounts"},
        # 4. Snapshot intermediate state
        {"store": "normalized_orders"},
        # 5. Standard transformer again
        {"transformer": "Filter",
         "params": {"input_col": "region", "perform": "keep",
                    "operator": "eq", "value": "US"}},
        # 6. Stamp audit columns before writing
        {"function": "add_audit_cols",
         "kwargs": {"run_id": "run_20260227_001"},
         "description": "Append audit metadata"},
    ]
}

pipe = load_pipeline(
    config,
    extra_functions={
        "normalize_amounts": normalize_amounts,
        "add_audit_cols": add_audit_cols,
    }
)
pipe.show(add_params=True)

result = pipe.run(orders)
print(result)
print(f"\nStored 'normalized_orders': {ns.get('normalized_orders').shape}")

2026-03-09 12:46:43,887 | [INFO]: Starting pipeline 'Order ETL' 
2026-03-09 12:46:43,890 | [INFO]: Running 'AssertNotEmpty' ... 
2026-03-09 12:46:43,890 | [INFO]: Completed 'AssertNotEmpty' in 0.0s 
2026-03-09 12:46:43,891 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:43,892 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:43,893 | [INFO]: Running 'normalize_amounts' ... 
2026-03-09 12:46:43,895 | [INFO]: Completed 'normalize_amounts' in 0.0s 
2026-03-09 12:46:43,895 | [INFO]:    --> Store df with key "normalized_orders" 
2026-03-09 12:46:43,896 | [INFO]: Nebula Storage: setting an object (<class 'polars.dataframe.frame.DataFrame'>) with the key "normalized_orders". 
2026-03-09 12:46:43,896 | [INFO]: Running 'Filter' ... 
2026-03-09 12:46:43,897 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:46:43,898 | [INFO]: Running 'add_audit_cols' ... 
2026-03-09 12:46:43,899 | [INFO]: Completed 'add_audit_cols' in 0.0s 
2026-03-09 12:46:43,899 | [INFO]: Pipeline 'Order

*** Order ETL *** (5 transformations)
 - AssertNotEmpty
 - SelectColumns -> PARAMS: columns=['order_id', 'customer', 'amount', 'region']
 - normalize_amounts -> KWARGS={'decimals': 2}
     Description: Round & cap amounts
   --> Store df with key "normalized_orders"
 - Filter -> PARAMS: input_col="region", operator="eq", perform="keep", value="US"
 - add_audit_cols -> KWARGS={'run_id': 'run_20260227_001'}
     Description: Append audit metadata
shape: (2, 6)
┌──────────┬──────────┬────────┬────────┬──────────────────┬────────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ run_id           ┆ load_date  │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---              ┆ ---        │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str              ┆ str        │
╞══════════╪══════════╪════════╪════════╪══════════════════╪════════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ run_20260227_001 ┆ 2026-03-09 │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ run_20260227_001 ┆ 2026-03-09 │
└──────────┴───────

---
## Part 4: Lazy Parameters (`__ns__`)

For runtime parameter resolution, use `lazy: true` with the `__ns__` prefix to reference values from nebula storage.

See notebook 04 for `LazyWrapper` details.

In [10]:
ns.clear()
ns.set("computed_label", "premium_customer")

config = {
    "pipeline": [
        {
            "transformer": "AddLiterals",
            "lazy": True,  # Enable lazy resolution
            "params": {
                "data": [
                    {"alias": "static_col", "value": "hardcoded"},
                    {"alias": "dynamic_col", "value": "__ns__computed_label"},  # Resolved at runtime
                ]
            }
        }
    ]
}

pipe = load_pipeline(config)
result = pipe.run(orders.head(2))
print(result)

2026-03-09 12:46:43,916 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:46:43,917 | [INFO]: Nebula Storage: 0 keys remained after clearing. 
2026-03-09 12:46:43,918 | [INFO]: Nebula Storage: setting an object (<class 'str'>) with the key "computed_label". 
2026-03-09 12:46:43,919 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,919 | [INFO]: Running '(Lazy) AddLiterals' ... 
2026-03-09 12:46:43,920 | [INFO]: Completed '(Lazy) AddLiterals' in 0.0s 
2026-03-09 12:46:43,921 | [INFO]: Pipeline completed in 0.0s 


shape: (2, 6)
┌──────────┬──────────┬────────┬────────┬────────────┬──────────────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ static_col ┆ dynamic_col      │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---        ┆ ---              │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str        ┆ str              │
╞══════════╪══════════╪════════╪════════╪════════════╪══════════════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ hardcoded  ┆ premium_customer │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ hardcoded  ┆ premium_customer │
└──────────┴──────────┴────────┴────────┴────────────┴──────────────────┘


The `__ns__` prefix works at any nesting depth within `params`.

---
## Part 5: Loops (Dynamic Expansion)

Loops let you generate repetitive pipeline sections dynamically. The `loop` block expands at load time, creating multiple transformers or pipelines from a template.

### 5.1 Basic Loop

In [11]:
config = {
    "pipeline": [
        {
            "loop": {
                "values": {
                    "col_name": ["flag_a", "flag_b", "flag_c"]
                },
                "transformer": "AddLiterals",
                "params": {
                    "data": [{"alias": "<<col_name>>", "value": True}]
                }
            }
        }
    ]
}

pipe = load_pipeline(config)
pipe.show(add_params=True)

*** Pipeline *** (3 transformations)
 - AddLiterals -> PARAMS: data=[{'alias': 'flag_a', 'value': True}]
 - AddLiterals -> PARAMS: data=[{'alias': 'flag_b', 'value': True}]
 - AddLiterals -> PARAMS: data=[{'alias': 'flag_c', 'value': True}]


In [12]:
result = pipe.run(orders.head(2))
print(result)

2026-03-09 12:46:43,946 | [INFO]: Starting pipeline 
2026-03-09 12:46:43,946 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:43,946 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:43,946 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:43,951 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:43,951 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:43,951 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:43,953 | [INFO]: Pipeline completed in 0.0s 


shape: (2, 7)
┌──────────┬──────────┬────────┬────────┬────────┬────────┬────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ flag_a ┆ flag_b ┆ flag_c │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ bool   ┆ bool   ┆ bool   │
╞══════════╪══════════╪════════╪════════╪════════╪════════╪════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ true   ┆ true   ┆ true   │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ true   ┆ true   ┆ true   │
└──────────┴──────────┴────────┴────────┴────────┴────────┴────────┘


### 5.2 Loop Modes: Linear vs Product

| Mode | Description |
|------|-------------|
| `linear` | Zip values together (default) |
| `product` | Cartesian product of all values |

In [13]:
# Linear mode: zip(["a", "b"], [1, 2]) → [("a", 1), ("b", 2)]
config = {
    "pipeline": [
        {
            "loop": {
                "mode": "linear",
                "values": {
                    "name": ["col_x", "col_y"],
                    "val": [100, 200]
                },
                "transformer": "AddLiterals",
                "params": {
                    "data": [{"alias": "<<name>>", "value": "<<val>>"}]
                }
            }
        }
    ]
}

pipe = load_pipeline(config)
pipe.show(add_params=True)
# Creates: AddLiterals(col_x=100), AddLiterals(col_y=200)

*** Pipeline *** (2 transformations)
 - AddLiterals -> PARAMS: data=[{'alias': 'col_x', 'value': 100}]
 - AddLiterals -> PARAMS: data=[{'alias': 'col_y', 'value': 200}]


In [14]:
# Product mode: product(["a", "b"], [1, 2]) → [("a", 1), ("a", 2), ("b", 1), ("b", 2)]
config = {
    "pipeline": [
        {
            "loop": {
                "mode": "product",
                "values": {
                    "prefix": ["US", "EU"],
                    "metric": ["sales", "returns"]
                },
                "transformer": "AddLiterals",
                "params": {
                    "data": [{"alias": "<<prefix>>_<<metric>>", "value": 0}]
                }
            }
        }
    ]
}

pipe = load_pipeline(config)
pipe.show(add_params=True)
# Creates 4 transformers: US_sales, US_returns, EU_sales, EU_returns

*** Pipeline *** (4 transformations)
 - AddLiterals -> PARAMS: data=[{'alias': 'US_sales', 'value': 0}]
 - AddLiterals -> PARAMS: data=[{'alias': 'US_returns', 'value': 0}]
 - AddLiterals -> PARAMS: data=[{'alias': 'EU_sales', 'value': 0}]
 - AddLiterals -> PARAMS: data=[{'alias': 'EU_returns', 'value': 0}]


### 5.3 Loop Over Pipelines

Loops can generate entire pipelines (with branches):

In [15]:
ns.clear()

config = {
    "pipeline": [
        {"store": "original"},  # Store for branch to read
        {
            "loop": {
                "mode": "linear",
                "values": {
                    "col_name": ["metric_a", "metric_b"],
                    "col_value": [10, 20]
                },
                "branch": {
                    "storage": "original",
                    "end": "join",
                    "on": "order_id",
                    "how": "left"
                },
                "pipeline": [
                    {"transformer": "SelectColumns", "params": {"columns": ["order_id"]}},
                    {"transformer": "AddLiterals", "params": {
                        "data": [{"alias": "<<col_name>>", "value": "<<col_value>>"}]
                    }}
                ]
            }
        }
    ]
}

pipe = load_pipeline(config)
pipe.show(add_params=True)

2026-03-09 12:46:43,997 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:46:43,997 | [INFO]: Nebula Storage: 0 keys remained after clearing. 


*** Pipeline *** (4 transformations)
   --> Store df with key "original"
*** Pipeline *** (2 transformations)
------ BRANCH (from storage: original) ------
>> Branch (2 transformations):
     - SelectColumns -> PARAMS: columns=['order_id']
     - AddLiterals -> PARAMS: data=[{'alias': 'metric_a', 'value': 10}]
<<< Join DFs >>>
  - how: left
  - on: order_id
*** Pipeline *** (2 transformations)
------ BRANCH (from storage: original) ------
>> Branch (2 transformations):
     - SelectColumns -> PARAMS: columns=['order_id']
     - AddLiterals -> PARAMS: data=[{'alias': 'metric_b', 'value': 20}]
<<< Join DFs >>>
  - how: left
  - on: order_id


In [16]:
result = pipe.run(orders.head(3))
print(result)

2026-03-09 12:46:44,011 | [INFO]: Starting pipeline 
2026-03-09 12:46:44,012 | [INFO]:    --> Store df with key "original" 
2026-03-09 12:46:44,012 | [INFO]: Nebula Storage: setting an object (<class 'polars.dataframe.frame.DataFrame'>) with the key "original". 
2026-03-09 12:46:44,013 | [INFO]: Entering branch 
2026-03-09 12:46:44,013 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:44,015 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:44,015 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,015 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:44,018 | [INFO]: Entering branch 
2026-03-09 12:46:44,018 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:44,018 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:44,018 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,018 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:44,023 | [INFO]: Pipeline completed in 0.0s 


shape: (3, 6)
┌──────────┬──────────┬────────┬────────┬──────────┬──────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ metric_a ┆ metric_b │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---      ┆ ---      │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ i32      ┆ i32      │
╞══════════╪══════════╪════════╪════════╪══════════╪══════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ 10       ┆ 20       │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ 10       ┆ 20       │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ 10       ┆ 20       │
└──────────┴──────────┴────────┴────────┴──────────┴──────────┘


### 5.4 Skip/Perform with Loops

In [17]:
config = {
    "pipeline": [
        {
            "skip": True,  # Entire loop is skipped
            "loop": {
                "values": {"x": [1, 2, 3]},
                "transformer": "AssertNotEmpty"  # Never runs
            }
        },
        {"transformer": "SelectColumns", "params": {"glob": "*"}}
    ]
}

pipe = load_pipeline(config)
pipe.show()  # Only SelectColumns appears

*** Pipeline *** (1 transformation)
 - SelectColumns


---
## Part 6: Complete Examples

### 6.1 Split Pipeline in Config

In [18]:
def split_by_value(df):
    return {
        "high": df.filter(pl.col("amount") >= 150),
        "low": df.filter(pl.col("amount") < 150),
    }


config = {
    "name": "Value-Based Processing",
    "split_function": "split_by_value",
    "split_order": ["high", "low"],
    "pipeline": {
        "high": [
            {"transformer": "AddLiterals", "params": {
                "data": [{"alias": "priority", "value": "high"}]
            }}
        ],
        "low": [
            {"transformer": "AddLiterals", "params": {
                "data": [{"alias": "priority", "value": "standard"}]
            }}
        ]
    }
}

pipe = load_pipeline(config, extra_functions={"split_by_value": split_by_value})
pipe.show()

result = pipe.run(orders)
print(result)

2026-03-09 12:46:44,058 | [INFO]: Starting pipeline 'Value-Based Processing' 
2026-03-09 12:46:44,059 | [INFO]: Entering split 
2026-03-09 12:46:44,060 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,061 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:44,062 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,063 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:44,065 | [INFO]: Pipeline 'Value-Based Processing' completed in 0.0s 


*** Value-Based Processing *** (2 transformations)
------ SPLIT ------ (function: split_by_value)
**SPLIT <<< high >>> (1 transformation):
     - AddLiterals
**SPLIT <<< low >>> (1 transformation):
     - AddLiterals
<<< Append DFs >>>
shape: (5, 5)
┌──────────┬──────────┬────────┬────────┬──────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ priority │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---      │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str      │
╞══════════╪══════════╪════════╪════════╪══════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ high     │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ high     │
│ 5        ┆ bob      ┆ 300.0  ┆ EU     ┆ high     │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ standard │
│ 4        ┆ carol    ┆ 50.0   ┆ APAC   ┆ standard │
└──────────┴──────────┴────────┴────────┴──────────┘


### 6.2 Branch Pipeline in Config

In [19]:
ns.clear()

# Pre-store customer tiers
tiers = pl.DataFrame({
    "customer": ["alice", "bob", "carol"],
    "tier": ["gold", "silver", "bronze"]
})
ns.set("customer_tiers", tiers)

config = {
    "name": "Enrich with Customer Tier",
    "branch": {
        "storage": "customer_tiers",
        "end": "join",
        "on": "customer",
        "how": "left"
    },
    "pipeline": [
        {"transformer": "SelectColumns", "params": {"columns": ["customer", "tier"]}}
    ]
}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders)
print(result)

2026-03-09 12:46:44,089 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:46:44,090 | [INFO]: Nebula Storage: 0 keys remained after clearing. 
2026-03-09 12:46:44,091 | [INFO]: Nebula Storage: setting an object (<class 'polars.dataframe.frame.DataFrame'>) with the key "customer_tiers". 
2026-03-09 12:46:44,092 | [INFO]: Starting pipeline 'Enrich with Customer Tier' 
2026-03-09 12:46:44,093 | [INFO]: Entering branch 
2026-03-09 12:46:44,093 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:46:44,094 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:46:44,096 | [INFO]: Pipeline 'Enrich with Customer Tier' completed in 0.0s 


*** Enrich with Customer Tier *** (1 transformation)
------ BRANCH (from storage: customer_tiers) ------
>> Branch (1 transformation):
     - SelectColumns
<<< Join DFs >>>
shape: (5, 5)
┌──────────┬──────────┬────────┬────────┬────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ tier   │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---    │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str    │
╞══════════╪══════════╪════════╪════════╪════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ gold   │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ silver │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ gold   │
│ 4        ┆ carol    ┆ 50.0   ┆ APAC   ┆ bronze │
│ 5        ┆ bob      ┆ 300.0  ┆ EU     ┆ silver │
└──────────┴──────────┴────────┴────────┴────────┘


### 6.3 Apply-to-Rows in Config

In [20]:
config = {
    "name": "Discount High-Value Orders",
    "apply_to_rows": {
        "input_col": "amount",
        "operator": "ge",
        "value": 200
    },
    "pipeline": [
        {"transformer": "AddLiterals", "params": {
            "data": [{"alias": "discount", "value": 0.10}]
        }}
    ],
    "otherwise": [
        {"transformer": "AddLiterals", "params": {
            "data": [{"alias": "discount", "value": 0.0}]
        }}
    ]
}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders)
print(result)

2026-03-09 12:46:44,105 | [INFO]: Starting pipeline 'Discount High-Value Orders' 
2026-03-09 12:46:44,106 | [INFO]: Entering apply_to_rows 
2026-03-09 12:46:44,110 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,111 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:46:44,112 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:46:44,113 | [INFO]: Completed 'AddLiterals' in 0.0s 


*** Discount High-Value Orders *** (2 transformations)
------ APPLY TO ROWS (amount ge 200) ------
>> Apply To rows (1 transformation):
     - AddLiterals
>> Otherwise (1 transformation)
     - AddLiterals
<<< Append DFs >>>


2026-03-09 12:46:44,115 | [INFO]: Pipeline 'Discount High-Value Orders' completed in 0.0s 


shape: (5, 5)
┌──────────┬──────────┬────────┬────────┬──────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ discount │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---      │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ f64      │
╞══════════╪══════════╪════════╪════════╪══════════╡
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ 0.1      │
│ 5        ┆ bob      ┆ 300.0  ┆ EU     ┆ 0.1      │
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ 0.0      │
│ 2        ┆ bob      ┆ 75.0   ┆ EU     ┆ 0.0      │
│ 4        ┆ carol    ┆ 50.0   ┆ APAC   ┆ 0.0      │
└──────────┴──────────┴────────┴────────┴──────────┘


---
## Summary

| Feature | Syntax |
|---------|--------|
| Basic transformer | `{"transformer": "Name", "params": {...}}` |
| Skip/disable | `"skip": true` or `"perform": false` |
| Description | `"description": "Human-readable text"` |
| Lazy params | `"lazy": true` + `"__ns__key"` in params |
| Storage | `{"store": "key"}`, `{"store_debug": "key"}`, `{"from_store": "key"}` |
| Eagerness | `"to_lazy"` (eager → lazy), `"collect"` (lazy → eager) |
| Loops | `{"loop": {"values": {...}, "transformer": ...}}` |
| Custom transformers | `extra_transformers={"Name": Class}` |
| Custom functions | `extra_functions={"function": func_or_name, "args": [...], "kwargs": {...}}` |

**Best practices:**
- Use `dict[str, T]` for extras (refactor-safe)
- Keep config keys stable, even if Python names change
- Use `skip`/`perform` for feature flags
- Organize custom transformers in modules with `__all__`
- Pair `"to_lazy"` + `"collect"` to bracket a lazy optimization zone

In [21]:
ns.clear()

2026-03-09 12:46:44,137 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:46:44,138 | [INFO]: Nebula Storage: 0 keys remained after clearing. 
